In [ ]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="") #свій ключ з робофлоу
project = rf.workspace("bohdans-workspace-gnare").project("kpi_yolo_traffic")
version = project.version(1)
dataset = version.download("yolov11")

loading Roboflow workspace...
loading Roboflow project...


In [ ]:
# Підключаю Google Drive для перманентного зберігання результатів навчання та вагів моделі.
# Це дозволить уникнути втрати даних після завершення сесії у Google Colab.
from google.colab import drive
drive.mount('/content/drive')

# Встановлюю бібліотеку Ultralytics, яка містить необхідні інструменти для роботи з архітектурою YOLOv11.
!pip install ultralytics

from ultralytics import YOLO
import os

# Ініціалізую базову модель YOLOv11.
# Обираю архітектуру 'nano' (yolo11n.pt) для досягнення оптимального балансу між швидкістю обробки відеопотоку та точністю детекції.
model = YOLO("yolo11n.pt")

# Запускаю процес навчання моделі на підготовленому наборі даних.
# Параметри навчання:
# - data: шлях до конфігураційного файлу датасету.
# - epochs=50: кількість ітерацій навчання для стабілізації вагів.
# - imgsz=640: розмір вхідного зображення для нейромережі.
# - project: шлях до папки на Google Drive для автоматичного збереження логів та результатів.
results = model.train(
    data=f"{dataset.location}/data.yaml",
    epochs=50,
    imgsz=640,
    project="/content/drive/MyDrive/MKR_Traffic_Project",
    name="yolo11_model_v1"
)

# Після завершення навчання найкращі ваги будуть автоматично збережені у папку:
# /content/drive/MyDrive/MKR_Traffic_Project/yolo11_model_v1/weights/best.pt

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Ultralytics 8.4.38 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/KPI_YOLO_Traffic-1/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937,

In [ ]:
# 1. Підключаю Google Drive заново, оскільки сервер Colab був оновлений
from google.colab import drive
drive.mount('/content/drive')

# 2. Встановлюю бібліотеку Ultralytics у нове середовище
!pip install ultralytics

# 3. Ініціалізую архітектуру YOLO та одразу завантажую СВОЇ найкращі ваги з Диску,
# минаючи процес повторного навчання.
from ultralytics import YOLO
model = YOLO("/content/drive/MyDrive/MKR_Traffic_Project/yolo11_model_v1/weights/best.pt")

print("✅ Модель успішно завантажена з Диску і готова обробляти відео!")

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 72.6 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
✅ Модель успішно завантажена з Диску і готова обробляти відео!


In [ ]:
import os
import cv2
import albumentations as A
import glob
from tqdm import tqdm
# 2 ПУНКТ
# 1. Ініціалізую конвеєр аугментації (7 методів згідно з вимогами МКР)
transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(p=0.4),
    A.GaussNoise(var_limit=(10.0, 50.0), p=0.3),
    A.MotionBlur(blur_limit=5, p=0.3),
    A.ShiftScaleRotate(shift_limit=0.06, rotate_limit=15, p=0.4),
    A.CLAHE(clip_limit=4.0, p=0.3),
    A.HueSaturationValue(p=0.3)
], bbox_params=A.BboxParams(format='yolo', label_fields=['class_labels']))

# Знаходжу всі оригінальні зображення у папці train
train_images = glob.glob(f"{dataset.location}/train/images/*.jpg")
print(f"Оригінальна кількість фото для навчання: {len(train_images)}")

print("Розпочинаю генерацію нових зображень...")
for img_path in tqdm(train_images):
    label_path = img_path.replace('images', 'labels').replace('.jpg', '.txt')

    # Зчитую оригінальне зображення
    image = cv2.imread(img_path)
    if image is None: continue
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    # Зчитую оригінальні Bounding Boxes
    bboxes, class_labels = [], []
    if os.path.exists(label_path):
        with open(label_path, 'r') as f:
            for line in f:
                parts = line.split()
                class_labels.append(int(parts[0]))
                bboxes.append([float(x) for x in parts[1:]])

    # 2. Створюю 2 унікальні копії для кожного фото і зберігаю їх у ту ж папку
    for i in range(2):
        try:
            transformed = transform(image=image, bboxes=bboxes, class_labels=class_labels)

            # Формую нові імена файлів з префіксом aug
            new_img_name = img_path.replace('.jpg', f'_aug{i}.jpg')
            new_lbl_name = label_path.replace('.txt', f'_aug{i}.txt')

            # Зберігаю аугментоване зображення
            cv2.imwrite(new_img_name, cv2.cvtColor(transformed['image'], cv2.COLOR_RGB2BGR))

            # Зберігаю перераховані координати рамок
            with open(new_lbl_name, 'w') as f:
                for cls, bbox in zip(class_labels, transformed['bboxes']):
                    f.write(f"{cls} {' '.join(map(str, bbox))}\n")
        except Exception:
            pass # Ігнорую випадки, коли після трансформації рамка повністю зникає з кадру

# Перевіряю новий розмір датасету
new_train_images = glob.glob(f"{dataset.location}/train/images/*.jpg")
print(f"✅ Фізичну аугментацію завершено! Нова база для навчання: {len(new_train_images)} фото.")

Argument(s) 'var_limit' are not valid for transform GaussNoise
ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.


Оригінальна кількість фото для навчання: 557
Розпочинаю генерацію нових зображень...


100%|██████████| 557/557 [00:07<00:00, 76.72it/s]

✅ Фізичну аугментацію завершено! Нова база для навчання: 1111 фото.


In [ ]:
from ultralytics import YOLO
# 3 ПУНКТ
# Ініціалізую базову модель YOLOv11 для навчання на аугментованому наборі даних
model = YOLO("yolo11n.pt")

# Запускаю процес навчання.
# Використовую batch=16 для реалізації алгоритму пакетної обробки
# що дозволяє стабілізувати градієнт і прискорити обробку в порівнянні з одиничними прогонами.
results = model.train(
    data=f"{dataset.location}/data.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    project="/content/drive/MyDrive/MKR_Traffic_Project",
    name="yolo11_final_augmented" # Зберігаємо в нову папку, щоб не затерти старі результати
)

print("✅ Фінальне навчання на розширеному датасеті успішно розпочато!")

Ultralytics 8.4.39 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/KPI_YOLO_Traffic-1/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolo11_final_augmented, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap

In [ ]:
!pip install sahi

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.7/111.7 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.0/63.0 MB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 14.4 MB/s eta 0:00:00
  Attempting uninstall: opencv-python
    Found existing installation: opencv-python 4.13.0.92
    Uninstalling opencv-python-4.13.0.92:
      Successfully uninstalled opencv-python-4.13.0.92


In [ ]:
# 0. Диск вже підключено, але залишаємо для цілісності скрипта
from google.colab import drive
drive.mount('/content/drive')

# Імпортую необхідні бібліотеки для обробки відео, SAHI (Sliced Inference) та трекінгу
import os
import cv2
from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction
from ultralytics import YOLO
from tqdm import tqdm
# 4 ПУНКТ
# 1. Ініціалізую модель SAHI (НОВИЙ СИНТАКСИС from_pretrained)
# Використовую ваги ФІНАЛЬНОЇ аугментованої моделі для пошуку дрібних об'єктів.
sahi_model = AutoDetectionModel.from_pretrained(
    model_type='yolov8',
    model_path='/content/drive/MyDrive/MKR_Traffic_Project/yolo11_final_augmented/weights/best.pt',
    confidence_threshold=0.4,
    device="cuda:0"
)

# 2. Ініціалізую стандартну модель YOLOv11 для алгоритму відстеження (ByteTrack)
tracker_model = YOLO('/content/drive/MyDrive/MKR_Traffic_Project/yolo11_final_augmented/weights/best.pt')

# 3. Визначаю шляхи до файлів
input_video_path = '/content/drive/MyDrive/MKR_Traffic_Project/test_video.mp4'
output_video_path = '/content/drive/MyDrive/MKR_Traffic_Project/result_traffic_final.mp4'

cap = cv2.VideoCapture(input_video_path)

# правильні константи OpenCV
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(output_video_path, fourcc, fps, (width, height))

print(f"Починаю обробку відео: {total_frames} кадрів ({width}x{height})")
print("Застосовується A/B тестування на льоту: ByteTrack vs SAHI")

for _ in tqdm(range(total_frames)):
    ret, frame = cap.read()
    if not ret:
        break

    # ЕТАП 1: Відстеження траєкторії (Tracking) через стандартну YOLO
    # Вона намалює рамки і лінії руху для тих машин, які зможе побачити на повному кадрі
    tracker_results = tracker_model.track(
        source=frame,
        persist=True,
        tracker="bytetrack.yaml",
        verbose=False
    )
    annotated_frame = tracker_results[0].plot()

    # ЕТАП 2: SAHI (Slicing) - Знаходимо дрібні об'єкти, які пропустила базова модель
    result = get_sliced_prediction(
        frame,
        sahi_model,
        slice_height=640,
        slice_width=640,
        overlap_height_ratio=0.2,
        overlap_width_ratio=0.2,
        verbose=0
    )

    # Домальовуємо зелені рамки SAHI поверх кадру для наочного порівняння
    for pred in result.object_prediction_list:
        bbox = pred.bbox.to_xyxy()
        cv2.rectangle(annotated_frame, (int(bbox[0]), int(bbox[1])), (int(bbox[2]), int(bbox[3])), (0, 255, 0), 2)
        cv2.putText(annotated_frame, "SAHI", (int(bbox[0]), int(bbox[1])-5), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

    out.write(annotated_frame)

cap.release()
out.release()
print(f"\n✅ Фінальне відео успішно оброблено та збережено: {output_video_path}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Починаю обробку відео: 920 кадрів (2160x3840)
Застосовується A/B тестування на льоту: ByteTrack vs SAHI


  0%|          | 0/920 [00:00<?, ?it/s]

requirements: Ultralytics requirement ['lap>=0.5.12'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: /usr
Resolved 2 packages in 190ms
Prepared 1 package in 37ms
Installed 1 package in 7ms
 + lap==0.5.13

requirements: AutoUpdate success ✅ 0.7s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect



100%|██████████| 920/920 [20:27<00:00,  1.33s/it]


✅ Фінальне відео успішно оброблено та збережено: /content/drive/MyDrive/MKR_Traffic_Project/result_traffic_final.mp4
